# `decode_mplog_proto_dataframe` — example notebook

This notebook demonstrates the `decode_mplog_proto_dataframe` API added in `inference-logging-client` 0.3.4.

Use this method when:
- All rows in the input DataFrame are encoded as **proto** (not arrow / parquet).
- You already have the feature schema in hand (from your own API, a cached JSON, or a prior `get_feature_schema` call).
- You want to avoid contacting the inference service at decode time (no schema fetch, no positive cache, no negative cache, no per-worker fallback).

Compared to `decode_mplog_dataframe`, this method skips: driver-side `distinct().collect()` for schema discovery, per-row metadata-byte parsing, format dispatch, and the `schema_cache.get()` per row. Same distributed `mapInPandas` pipeline, same Arrow 2 GiB safety (default `max_records_per_batch=50`), same input-column projection.

## 1. Install

On Databricks, install at the cluster or notebook level:

In [ ]:
%pip install --upgrade inference-logging-client==0.3.4 zstandard

In [ ]:
dbutils.library.restartPython()  # Databricks only

## 2. Imports and Spark session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from inference_logging_client import decode_mplog_proto_dataframe

spark = SparkSession.builder.appName("decode_mplog_proto_example").getOrCreate()

## 3. Load the encoded logs DataFrame

Adjust the table name / path and the partition filter for your environment. The DataFrame must have at least a `features` column (the encoded payloads) and a `mp_config_id` column. Optional columns that get passed through if present: `entities`, `parent_entity`, `prism_ingested_at`, `prism_extracted_at`, `created_at`, `tracking_id`, `user_id`, `year`, `month`, `day`, `hour`.

In [ ]:
logs_df = (
    spark.table("silver.ML_Platform__model_proxy_inference_logs")
    .filter(F.col("mp_config_id") == "my-model-proxy-id")
    .filter(F.concat_ws("-", "year", "month", "day") == "2026-05-09")
)

logs_df.printSchema()
logs_df.limit(3).show(truncate=80)

## 4. Provide the feature schema

The `schema` argument accepts three shapes — pick whichever your source already produces:

**Option A — inference-service JSON response shape (most common):**
```python
schema = {"data": [{"feature_name": "...", "feature_type": "DataTypeFP32", "feature_size": 1}, ...]}
```

**Option B — plain list of dicts:**
```python
schema = [{"feature_name": "...", "feature_type": "DataTypeFP32"}, ...]
```

**Option C — typed `FeatureInfo` list (returned by `get_feature_schema`):**
```python
from inference_logging_client import get_feature_schema
schema = get_feature_schema("my-model-proxy-id", 1)
```

Array order is preserved and used as the proto field index — do not reorder.

In [ ]:
# Example: schema fetched from your own API, in the inference-service JSON shape.
schema = {
    "data": [
        {"feature_name": "user:derived_2_fp32:log_views_56day", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:orders_by_clicks_laplace_56day", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:avg_click_catalog_nqd_30day", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:avg_order__catalog_arp_sscat_percentile__90day", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:browse_time_last_7day", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:clicks_by_views_laplace_28day", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:engagement_click_percent", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:retention_90_days", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:user__nqp", "feature_type": "DataTypeFP32", "feature_size": 1},
        {"feature_name": "user:derived_2_fp32:user__nqp_by_nqd", "feature_type": "DataTypeFP32", "feature_size": 1},
    ]
}

print(f"schema has {len(schema['data'])} features")

## 5. Decode

Defaults that matter:
- `decompress=True` — automatically zstd-decompresses each payload if needed.
- `num_partitions=10000` — keeps each worker task small when rows carry multi-MB payloads.
- `max_records_per_batch=50` — keeps each Arrow batch under the 2 GiB per-column limit.
- `needed_columns=None` — decode all schema columns; pass a list/set to project early.

In [ ]:
decoded_df = decode_mplog_proto_dataframe(
    df=logs_df,
    spark=spark,
    schema=schema,
)

decoded_df.printSchema()

In [ ]:
decoded_df.limit(5).show(truncate=80)

## 6. Decode only the columns you need (optional)

Pass `needed_columns` so workers skip decoding and emitting features you don't care about. Significantly reduces output size and worker memory when the schema is wide.

In [ ]:
subset_df = decode_mplog_proto_dataframe(
    df=logs_df,
    spark=spark,
    schema=schema,
    needed_columns={
        "user:derived_2_fp32:log_views_56day",
        "user:derived_2_fp32:retention_90_days",
    },
)

subset_df.printSchema()
subset_df.limit(5).show(truncate=80)

## 7. Tuning knobs (optional)

Adjust if your rows are unusually small or unusually large:

In [ ]:
tuned_df = decode_mplog_proto_dataframe(
    df=logs_df,
    spark=spark,
    schema=schema,
    num_partitions=20000,        # raise if executors are CPU-starved
    max_records_per_batch=20,    # lower further if you still hit Arrow overflow
    decompress=True,
)

tuned_df.limit(3).show(truncate=80)

## 8. Persist the decoded output

In [ ]:
(
    decoded_df
    .write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet("s3://your-bucket/decoded_mplog/my-model-proxy-id/")
)

## Notes

- **Format is always PROTO.** If your logs use arrow or parquet encoding, use `decode_mplog_dataframe` instead.
- **Schema is applied to every row.** All rows in the input DataFrame must have been encoded against the schema you pass. If you have multiple `(mp_config_id, version)` combos in the same DataFrame and they use different schemas, filter and decode each group separately.
- **Type strings.** `DataTypeFP32`, `FP32`, `fp32` — all work. The decoder strips the `DataType` prefix and case-normalizes internally.
- **`feature_size`.** Ignored. The decoder infers scalar vs vector from the type name (`FP32` vs `FP32Vector`).